:important: This is the notebook for my Wildfires project for SDS210. Broad structure as follows:

1. Import required packages
    

2. test pull data into project with the FIRMS API and then just use the bounding box for Australia.

3. reproject to correct CRS




In [1]:
import requests
import pandas as pd
import geopandas as gpd
import time
import datetime
import folium
from folium.plugins import MarkerCluster
import numpy as np

In [2]:
# We need to access the API and to do that, will use the map key that permits access.
MAP_KEY = '54684dde74a099b139ddbbef0f621891'

# Now let's check how many results we have

url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  test_response = requests.get(url)
  test_data = response.json()
  test_df = pd.Series(data)
  display(test_df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

There is an issue with the query. 
Try in your browser: https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=54684dde74a099b139ddbbef0f621891


In [3]:
# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
sensor_data = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
test_df = pd.read_csv(sensor_data)
display(test_df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-20
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-20
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-20
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-19
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-19
8,GOES_NRT,2022-08-09,2026-05-20
9,BA_MODIS,2000-11-01,2026-02-01


In [4]:
# We are interested in the wildfires in Australia and so will select this information using a bounding box. Aus = 110 -55, 180 -10
modis_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/110,-50,160,-11/3'
modis_nrt_df = pd.read_csv(modis_nrt_url)

viirs_noaa20_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/110,-50,160,-11/3'
viirs_noaa20_nrt_df = pd.read_csv(viirs_noaa20_nrt_url)

viirs_noaa21_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA21_NRT/110,-50,160,-11/3'
viirs_noaa21_nrt_df = pd.read_csv(viirs_noaa21_nrt_url)

viirs_snpp_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/110,-50,160,-11/3'
viirs_snpp_nrt_df = pd.read_csv(viirs_snpp_nrt_url)

In [5]:
print(modis_nrt_df.iloc[100,:])
print(viirs_noaa20_nrt_df.iloc[100,:])
print(viirs_noaa21_nrt_df.iloc[100,:])
print(viirs_snpp_nrt_df.iloc[100,:])

latitude       -13.40639
longitude      131.61371
brightness        312.19
scan                1.39
track               1.17
acq_date      2026-05-18
acq_time             629
satellite           Aqua
instrument         MODIS
confidence            44
version           6.1NRT
bright_t31        292.09
frp                 9.81
daynight               D
Name: 100, dtype: object
latitude       -15.60877
longitude      145.04248
bright_ti4         335.6
scan                0.45
track               0.47
acq_date      2026-05-18
acq_time             421
satellite            N20
instrument         VIIRS
confidence             n
version           2.0NRT
bright_ti5        298.94
frp                 4.85
daynight               D
Name: 100, dtype: object
latitude       -15.41796
longitude      144.76384
bright_ti4        330.07
scan                0.49
track               0.49
acq_date      2026-05-18
acq_time             330
satellite            N21
instrument         VIIRS
confidence             n


In [6]:
# Create a funtion that checks time of aquisition and calculates time delta
def add_time_since_acq(df):
    df["acq_date"] = pd.to_datetime(df["acq_date"])   # Ensuring date is in datetime format
    df["acq_datetime"] = pd.to_datetime(
        df["acq_date"].astype(str) + df["acq_time"].astype(str).str.zfill(4),    # Turns all values into 4 digit HHHH format
        format = "%Y-%m-%d%H%M"
        ).dt.tz_localize('UTC')

    current_time = pd.Timestamp.now('UTC')
    df["time_since_detection"] = current_time - df["acq_datetime"]

    # Calculate hours since detection
    df["hours_since_detection"] = df["time_since_detection"].dt.total_seconds()/3600

    # Convert data types to string format so folium can take them
    df["acq_date_str"] = df["time_since_detection"].astype(str)
    df["time_since_detection_str"] = df["acq_date"].astype(str)
    df["acq_datetime_str"] = df["acq_datetime"].astype(str)

    # Drop datetime columns and "time_since_detection_str" as it is covered by "hours_since_detection".
    df = df.drop(columns=["time_since_detection", "time_since_detection_str", "acq_date", "acq_datetime"])


    return df

# Apply to each of our datasets

modis_nrt_df = add_time_since_acq(modis_nrt_df)
viirs_noaa20_nrt_df = add_time_since_acq(viirs_noaa20_nrt_df)
viirs_noaa21_nrt_df = add_time_since_acq(viirs_noaa21_nrt_df)
viirs_snpp_nrt_df = add_time_since_acq(viirs_snpp_nrt_df)

In [7]:
# Convert to GeoDataFrame (WGS84)
modis_nrt_gdf = gpd.GeoDataFrame(
    modis_nrt_df, 
    geometry=gpd.points_from_xy(
        modis_nrt_df["longitude"], 
        modis_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa20_nrt_gdf = gpd.GeoDataFrame(
    viirs_noaa20_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa20_nrt_df["longitude"], 
        viirs_noaa20_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa21_nrt_gdf = gpd.GeoDataFrame(
    viirs_noaa21_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa21_nrt_df["longitude"], 
        viirs_noaa21_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_snpp_nrt_gdf = gpd.GeoDataFrame(
    viirs_snpp_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_snpp_nrt_df["longitude"], 
        viirs_snpp_nrt_df["latitude"]
    ),
    crs="EPSG:4326")


print(f"Found {len(modis_nrt_gdf)} fire records.")
print(f"Found {len(viirs_noaa20_nrt_gdf)} fire records.")
print(f"Found {len(viirs_noaa21_nrt_gdf)} fire records.")
print(f"Found {len(viirs_snpp_nrt_gdf)} fire records.")

Found 490 fire records.
Found 2310 fire records.
Found 2862 fire records.
Found 1602 fire records.


In [8]:
# 1. Initialize the Folium map (The Control)
australia_bushfire_map = folium.Map(location=[-28.281828, 136.145401], zoom_start=5)

# 2. Add multiple datasets using GeoPandas

modis_nrt_gdf.explore(
    m=australia_bushfire_map, 
    column='hours_since_detection', 
    name='MODIS NRT',
    tooltip=['confidence'], 
    cmap='YlOrRd_r',
    style_kwds={'fillOpacity': 0.4, 'color': 'white', 'weight': 0.1},
    show=True
)

viirs_noaa20_nrt_gdf.explore(
    m=australia_bushfire_map, 
    column='hours_since_detection', 
    name='VIIRS NOAA20 NRT',
    tooltip=['confidence'], 
    cmap='YlOrRd_r',
    style_kwds={'fillOpacity': 0.4, 'color': 'white', 'weight': 0.1},
    legend=False,
    show=True
)

viirs_noaa21_nrt_gdf.explore(
    m=australia_bushfire_map, 
    column='hours_since_detection', 
    name='VIIRS NOAA21 NRT',
    tooltip=['confidence'], 
    cmap='YlOrRd_r',
    style_kwds={'fillOpacity': 0.4, 'color': 'white', 'weight': 0.1},
    legend=False,
    show=True
)

viirs_snpp_nrt_gdf.explore(
    m=australia_bushfire_map, 
    column='hours_since_detection', 
    name='VIIRS SNPP NRT',
    tooltip=['confidence'], 
    cmap='YlOrRd_r',
    style_kwds={'fillOpacity': 0.4, 'color': 'white', 'weight': 0.1},
    legend=False,
    show=True
)


# 3. Add Folium plugins back on top
folium.LayerControl().add_to(australia_bushfire_map)

australia_bushfire_map

In [9]:
# Saving map for viewer access
australia_bushfire_map.save("australia_bushfire_map.html")

Now we will create a choropleth map to show numbers of fire per Local Government Area (LGA). 

In [10]:
# Add LGAs data to the project

aus_lgas = gpd.read_file("data/ASGS_Ed3_Non_ABS_Structures_GDA2020_updated_2025/ASGS_Ed3_Non_ABS_Structures_GDA2020_updated_2025.gpkg",
                        layer="LGA_2025_AUST_GDA2020"
                        ).to_crs(epsg=4326)

aus_lgas.head()

,LGA_CODE_2025,LGA_NAME_2025,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,geometry
0,10050,Albury,1,New South Wales,AUS,Australia,305.6386,"MULTIPOLYGON (((146.86566 -36.07292, 146.8663 ..."
1,10180,Armidale,1,New South Wales,AUS,Australia,7809.4406,"MULTIPOLYGON (((152.38816 -30.52639, 152.38744..."
2,10250,Ballina,1,New South Wales,AUS,Australia,484.9692,"MULTIPOLYGON (((153.57106 -28.87381, 153.57106..."
3,10300,Balranald,1,New South Wales,AUS,Australia,21690.7493,"MULTIPOLYGON (((143.00433 -33.78164, 142.99952..."
4,10470,Bathurst,1,New South Wales,AUS,Australia,3817.8645,"MULTIPOLYGON (((149.84877 -33.52784, 149.84894..."


In [16]:

# 2. Spatial Join: Count fire detections within each LGA
modis_nrt_joined = gpd.sjoin(
    modis_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)

# Set join key
name_col = "LGA_NAME_2025_right" if "LGA_NAME_2025_right" in modis_nrt_joined.columns else "LGA_NAME_2025"

# Group by the lga name and count the occurrences
modis_counts = modis_nrt_joined.groupby(name_col).size().reset_index(name="modis_count")

# Rename join key to match
modis_counts.rename(columns={name_col: "LGA_NAME_2025"}, inplace=True)

# Keeping only the relevant columns
modis_counts = modis_counts[["LGA_NAME_2025", "modis_count"]]


# Merge the counts back into the main LGAs GeoDataFrame
aus_lgas = aus_lgas.merge(modis_counts, on="LGA_NAME_2025", how="left")

# Fill NAs with 0 so we can calculate fires per km^2
aus_lgas["modis_counts"] = aus_lgas["modis_counts"].fillna(0)

# Calculate the density (fire detections per square kilometer)
#aus_lgas["fire_density"] = (aus_lgas["modis_count"] / aus_lgas["AREA_ALBERS_SQKM"]).round(2)

aus_lgas.head()

MergeError: Passing 'suffixes' which cause duplicate columns {'modis_count_x', 'modis_count_y'} is not allowed.

In [12]:

# 2. Spatial Join: Count fire detections within each LGA
modis_nrt_joined = gpd.sjoin(
    modis_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)

viirs_noaa20_nrt_joined = gpd.sjoin(
    viirs_noaa20_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)

viirs_noaa21_nrt_joined = gpd.sjoin(
    viirs_noaa21_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)

viirs_snpp_nrt_joined = gpd.sjoin(
    viirs_snpp_nrt_gdf, 
    aus_lgas, 
    how="inner", 
    predicate="within"
)


name_col = "LGA_NAME_2025_right" if "LGA_NAME_2025_right" in modis_nrt_joined.columns else "LGA_NAME_2025"

# Group by the lga name and count the occurrences
modis_counts = modis_nrt_joined.groupby(name_col).size().reset_index(name="modis_count")
noaa20_counts = viirs_noaa20_nrt_joined.groupby(name_col).size().reset_index(name="noaa20_count")
noaa21_counts = viirs_noaa21_nrt_joined.groupby(name_col).size().reset_index(name="noaaa21_count")
snpp_counts = viirs_snpp_nrt_joined.groupby(name_col).size().reset_index(name="snpp_count")

# Rename join key to match
modis_counts.rename(columns={name_col: "LGA_NAME_2025"}, inplace=True)


modis_counts = modis_counts[["LGA_NAME_2025", "modis_count"]]



# Merge the counts back into the main quarters GeoDataFrame
aus_lgas = (aus_lgas
    .merge(modis_counts, on="LGA_NAME_2025", how="left")
    .merge(noaa20_counts, on="LGA_NAME_2025", how="left")
    .merge(noaa21_counts, on="LGA_NAME_2025", how="left")
    .merge(snpp_counts, on="LGA_NAME_2025", how="left")
           )

#.fillna({"modis_count": 0})

# Calculate the density (fire detections per square kilometer)
aus_lgas["modis_count", "noaa20_count", "noaaa21_count", "snpp_count"].sum("fire_count")
aus_lgas["fire_density"] = (aus_lgas["modis_count"] / aus_lgas["AREA_ALBERS_SQKM"]).round(2)

#aus_lgas.head()

KeyError: ('modis_count', 'noaa20_count', 'noaaa21_count', 'snpp_count')

In [ ]:
# Initialize the map
bushfire_choropleth = folium.Map(
    location=[-28.281828, 136.145401], zoom_start=5, tiles="CartoDB Positron No Labels"
)

# 1. Area Choropleth (Hidden by default using show=False)
folium.Choropleth(
    geo_data=quarters_4326,
    name="Quarter Area (km2)",
    data=quarters_4326,
    columns=["name", "area_km2"],
    key_on="feature.properties.name",
    fill_color="YlGn",
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name="Area in km2",
    show=False,  # Keeps the map clean on initial load
).add_to(bushfire_choropleth)

# 2. Density Choropleth (Visible by default)
folium.Choropleth(
    geo_data=quarters_4326,
    name="Parking Density (locations/km2)",
    data=quarters_4326,
    columns=["name", "parking_density"],
    key_on="feature.properties.name",
    fill_color="viridis",
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name="Bike Parking Locations per km2",
).add_to(bushfire_choropleth)

# 3. Add Interactive Tooltips via an Invisible GeoJson Layer
folium.GeoJson(
    quarters_4326,
    name="Interactive Tooltips",
    # Make the polygons completely transparent so they do not hide the choropleth colors
    style_function=lambda x: {"fillColor": "#ffffff00", "color": "#ffffff00"},
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "parking_density"],
        aliases=["Quarter:", "Density:"],
        localize=True,
    ),
).add_to(bushfire_choropleth)

# Add Layer Control and display
folium.LayerControl().add_to(bushfire_choropleth)

bushfire_choropleth